## Importando as bliotecas e configurando o ambiente

In [ ]:
import pandas as pd
import ast
pd.set_option('display.max_columns', None)

In [ ]:
movies = pd.read_csv('movies_metadata.csv')
movies.head(5)

## Tratando as linhas duplicadas

### Visualizando as linhas duplicadas

In [ ]:
movies['duplicados'] = movies.duplicated()
resultado = list(movies['duplicados'] == True)

movies.loc[resultado].sort_values(by='id', ascending=True)

### Removendo as linhas duplicadas

In [ ]:
movies.drop_duplicates(inplace=True)

In [ ]:
movies['duplicados'] = movies.duplicated()
resultado = list(movies['duplicados'] == True)

movies.loc[resultado].sort_values(by='id', ascending=True)

## Tratando os valores nulos

### Visualizando e excluindo valores nulos 

In [ ]:
# Como vou fazer algumas análises ao longo do tempo, irei retirar as linhas que possuem a data de lançamento nula
movies['nulos'] = movies['release_date'].isnull()
nulos = list(movies['nulos'] == True)

movies.loc[nulos]

In [ ]:
movies.dropna(axis= 0, subset=['release_date'], inplace=True)


In [ ]:
#A mesma coisa com os valores de receita nulos
movies['nulos'] = movies['revenue'].isnull()
nulos = list(movies['nulos'] == True)
movies.loc[nulos]

In [ ]:
movies.dropna(axis= 0, subset=['revenue'], inplace=True)

In [ ]:
#Na primeira vez que eu importei os dados para o PowerBI o programa me retornou um erro porque algumas linhas estão com um formato de data na coluna do ID
#Provavelmente algum erro de salvamento, vou encontrar e excluir essas linhas.
# regex = r'\b\d{4}-\d{2}-\d{2}\b'

# movies['contains'] = movies['id'].str.contains(regex, regex=True)

# lista = list(movies['contains'] == True)

# movies.loc[lista]

## Criação tabelas auxiliares

Como os campos *genres*, *production_country* e *production_companies* da tabela movies_metada e o *keywords* da tabela keywords , estão em um formato similar a um JSON com vários valores, senti a necessidade de criar tabelas auxiliares para fazer as análise. Além disso, como um filme por exemplo, pode ter vários generos e um gênero pode estar presente em vários filmes, dessas relações N:N surgiu também uma tabela intemediária

### Criação das tabelas: dim_genres e dim_movies_genres

#### Definindo a função para tratar os campos

Como os valores no campo *genres* estão sempre no formato `'[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}]'`, a função ast.literal_eval vai ser responsável por reconher esse valores como estruturas de dados no python.

In [ ]:
def parse_genres(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        if not isinstance(genres, (list, tuple)):
            return []
        return [(g['id'], g['name']) for g in genres]
    except (ValueError, SyntaxError):
        return []
    
movies['parsed_genres'] = movies['genres'].apply(parse_genres)

#### Criação da tabela com todos os gêneros encontrados, sem repetição.

In [ ]:
all_genres = set()
for genres in movies['parsed_genres']:
    all_genres.update(genres)

genres_df = pd.DataFrame(list(all_genres), columns=['genre_id', 'genre_name'])

#### Criação da tabela intermediária

In [ ]:
movie_genre_pairs = []
for idx, row in movies.iterrows():
    for genre_id, genre_name in row['parsed_genres']:
        movie_genre_pairs.append((row['id'], genre_id))

movies_genres_df = pd.DataFrame(movie_genre_pairs, columns=['movie_id', 'genre_id'])

#### Salvando tabelas criadas

In [ ]:
movies_genres_df.to_csv('dim_movies_genres.csv', index=False)
genres_df.to_csv('dim_genres.csv', index=False)

### Criação das tabelas: dim_movies_countries e dim_movies_countries

In [ ]:
def parse_countries(country_str):
    try:
        item = ast.literal_eval(country_str)
        if not isinstance(item, (list, tuple)):
            return []
        return [(i["iso_3166_1"], i["name"]) for i in item]
    except (ValueError, SyntaxError):
        return []
    

# Aplicar a função
movies['parsed_countries'] = movies['production_countries'].apply(parse_countries)

# Agora construir tabelas:
# 1. Tabela de países únicos
all_countries = set()
for countries in movies['parsed_countries']:
    all_countries.update(countries)

countries_df = pd.DataFrame(list(all_countries), columns=['country_abr', 'country_name'])

# 2. Tabela de relacionamento filme-países
movie_countries_pairs = []
for idx, row in movies.iterrows():
    for country_abr, country_name in row['parsed_countries']:
        movie_countries_pairs.append((row['id'], country_abr))

movies_countries_df = pd.DataFrame(movie_countries_pairs, columns=['movie_id', 'country_abr'])

# 3. Salvando

movies_countries_df.to_csv('dim_movies_countries.csv', index=False)
countries_df.to_csv('dim_countries.csv', index=False)

### Criação das tabelas: dim_production_companies e dim_movies_production_companies

In [ ]:
# Função para tratar o campo de genres
def parse_production_companies(companies_str):
    try:
        item = ast.literal_eval(companies_str)
        if not isinstance(item, (list, tuple)):
            return []
        return [(i["id"], i["name"]) for i in item]
    except (ValueError, SyntaxError):
        return []
    

# Aplicar a função
movies['parsed_production_companies'] = movies['production_companies'].apply(parse_production_companies)

# Agora construir tabelas:
# 1. Tabela de produtores únicos
all_production_companies = set()
for production_companies in movies['parsed_production_companies']:
    all_production_companies.update(production_companies)

production_companies_df = pd.DataFrame(list(all_production_companies), columns=['production_companies_id', 'production_companies_name'])

# 2. Tabela de relacionamento filme-produtores
movie_production_companies_pairs = []
for idx, row in movies.iterrows():
    for country_abr, country_name in row['parsed_production_companies']:
        movie_production_companies_pairs.append((row['id'], country_abr))

movies_production_companies_df = pd.DataFrame(movie_production_companies_pairs, columns=['movie_id', 'production_companies_id'])

# 3. Salvando

movies_production_companies_df.to_csv('dim_movies_production_companies.csv', index=False)
production_companies_df.to_csv('dim_production_companies.csv', index=False)

### Criação das tabelas: dim_keywords e dim_movies_keywords

In [ ]:
# Função para tratar do campo
keywords_df = pd.read_csv('keywords.csv')

def parse_keywords(keywords_str):
    try:
        item = ast.literal_eval(keywords_str)
        if not isinstance(item, (list, tuple)):
            return []
        return [(i["id"], i["name"]) for i in item]
    except (ValueError, SyntaxError):
        return []
    

# Aplicar a função
keywords_df['parsed_keywords'] = keywords_df['keywords'].apply(parse_keywords)

# Agora construir tabelas:
# 1. Tabela de gêneros únicos
all_keywords = set()
for keywords in keywords_df['parsed_keywords']:
    all_keywords.update(keywords)

countries_df = pd.DataFrame(list(all_keywords), columns=['keywords_id', 'keywords_name'])

# 2. Tabela de relacionamento filme-gênero
movie_keywords_pairs = []
for idx, row in keywords_df.iterrows():
    for keywords_id, keywords_name in row['parsed_keywords']:
        movie_keywords_pairs.append((row['id'], keywords_id))

movies_keywords_df = pd.DataFrame(movie_keywords_pairs, columns=['movie_id', 'keywords_id'])

# 3. movies já é sua tabela de filmes

movies_keywords_df.to_csv('dim_movies_keywords.csv', index=False)
countries_df.to_csv('dim_keywords.csv', index=False)

## Limpando as colunas

In [ ]:
movies.drop(['belongs_to_collection','genres', 'homepage', 'poster_path', 'production_companies', 'production_countries', 'spoken_languages', 'duplicados', 'nulos', 'parsed_countries', 'parsed_genres', 'parsed_genres','parsed_production_companies'], axis = 1, inplace=True)
movies.head(5)

## Salvando o dataset

In [ ]:
movies.to_csv("fact_movies_dataset.csv", index=False)